In [1]:
from pyspark.sql import DataFrame, functions as F
from datetime import datetime
 
SILVER_SCHEMA = "silver"
DQ_RESULTS_TABLE = "dq_test_results"
 
EMAIL_REGEX = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
 

StatementMeta(, 4da8fec4-7f89-4805-b427-468307bcaa2d, 3, Finished, Available, Finished, False)

In [2]:
 
PRIMARY_KEYS = {
    "customers_t": "customer_id",
    "employees_t": "employee_id",
    "orders_t": "order_id",
    "order_items_t": "order_item_id",
    "products_t": "product_id",
    "stores_t": "store_id",
}
 
DQ_TEST_METADATA = []
 
for table_name, pk_column in PRIMARY_KEYS.items():
    DQ_TEST_METADATA.append({
        "table": table_name,
        "test_name": f"{pk_column}_not_null",
        "test_type": "not_null",
        "column": pk_column,
    })
    DQ_TEST_METADATA.append({
        "table": table_name,
        "test_name": f"{pk_column}_unique",
        "test_type": "unique",
        "column": pk_column,
    })
 
DQ_TEST_METADATA.extend([
    {"table": "products_t", "test_name": "product_price_non_negative",
     "test_type": "threshold", "column": "price", "operator": ">=", "threshold": 0},
 
    {"table": "employees_t", "test_name": "employee_salary_non_negative",
     "test_type": "threshold", "column": "salary", "operator": ">=", "threshold": 0},
 
    {"table": "orders_t", "test_name": "order_total_amount_non_negative",
     "test_type": "threshold", "column": "total_amount", "operator": ">=", "threshold": 0},
 
    {"table": "order_items_t", "test_name": "order_item_quantity_positive",
     "test_type": "threshold", "column": "quantity", "operator": ">", "threshold": 0},
    {"table": "order_items_t", "test_name": "order_item_unit_price_non_negative",
     "test_type": "threshold", "column": "unit_price", "operator": ">=", "threshold": 0},
    {"table": "order_items_t", "test_name": "order_item_line_amount_non_negative",
     "test_type": "threshold", "column": "line_amount", "operator": ">=", "threshold": 0},
 
    {"table": "customers_t", "test_name": "customer_email_not_null",
     "test_type": "not_null", "column": "email"},
    {"table": "customers_t", "test_name": "customer_email_format_valid",
     "test_type": "email_format", "column": "email"},
])
 

StatementMeta(, 4da8fec4-7f89-4805-b427-468307bcaa2d, 4, Finished, Available, Finished, False)

In [3]:
def test_not_null(df: DataFrame, column: str) -> int:
    return df.filter(F.col(column).isNull()).count()
 
 
def test_unique(df: DataFrame, column: str) -> int:
  
    dup_groups = (
        df.groupBy(column)
          .count()
          .filter(F.col("count") > 1)
    )
    result = dup_groups.agg(F.sum("count").alias("total")).collect()[0]["total"]
    return int(result) if result is not None else 0
 
 
def test_positive_value(df: DataFrame, column: str, operator: str = ">=",
                         threshold: float = 0, allow_null: bool = False) -> int:
    if operator == ">=":
        condition_met = F.col(column) >= threshold
    elif operator == ">":
        condition_met = F.col(column) > threshold
    elif operator == "<=":
        condition_met = F.col(column) <= threshold
    elif operator == "<":
        condition_met = F.col(column) < threshold
    else:
        raise ValueError(f"Unsupported operator '{operator}' for test_positive_value.")
 
    if allow_null:
        failing = df.filter(F.col(column).isNotNull() & ~condition_met)
    else:
        failing = df.filter(F.col(column).isNull() | ~condition_met)
 
    return failing.count()
 
 
def test_email_format(df: DataFrame, column: str) -> int:
    return df.filter(
        F.col(column).isNotNull() & ~F.col(column).rlike(EMAIL_REGEX)
    ).count()
 
TEST_DISPATCH = {
    "not_null": lambda df, t: test_not_null(df, t["column"]),
    "unique": lambda df, t: test_unique(df, t["column"]),
    "threshold": lambda df, t: test_positive_value(
        df, t["column"], t.get("operator", ">="), t.get("threshold", 0), t.get("allow_null", False)
    ),
    "email_format": lambda df, t: test_email_format(df, t["column"]),
}

StatementMeta(, 4da8fec4-7f89-4805-b427-468307bcaa2d, 5, Finished, Available, Finished, False)

In [4]:
def run_single_test(df: DataFrame, test_meta: dict) -> dict:
    test_type = test_meta["test_type"]
    if test_type not in TEST_DISPATCH:
        raise ValueError(
            f"Unknown test_type '{test_type}' for test '{test_meta['test_name']}'. "
            f"Valid types: {list(TEST_DISPATCH.keys())}"
        )
 
    failed_count = TEST_DISPATCH[test_type](df, test_meta)
    status = "PASS" if failed_count == 0 else "FAIL"
 
    return {
        "table_name": test_meta["table"],
        "test_name": test_meta["test_name"],
        "failed_record_count": int(failed_count),
        "status": status,
        "execution_timestamp": datetime.now(),
    }
 
 
def run_all_tests(test_metadata: list, schema: str = SILVER_SCHEMA) -> list:
    table_cache = {}
    results = []
 
    for test_meta in test_metadata:
        table_name = test_meta["table"]
 
        if table_name not in table_cache:
            full_name = f"{schema}.{table_name}"
            print(f"[READ] {full_name}")
            table_cache[table_name] = spark.read.table(full_name)
 
        df = table_cache[table_name]
        result = run_single_test(df, test_meta)
        print(f"[TEST] {result['table_name']}.{result['test_name']}: "
              f"{result['status']} (failed_record_count={result['failed_record_count']})")
        results.append(result)
 
    return results

StatementMeta(, 4da8fec4-7f89-4805-b427-468307bcaa2d, 6, Finished, Available, Finished, False)

In [5]:
dq_results = run_all_tests(DQ_TEST_METADATA, schema=SILVER_SCHEMA)
 
summary_df = spark.createDataFrame(dq_results).select(
    "table_name", "test_name", "failed_record_count", "status", "execution_timestamp"
)

StatementMeta(, 4da8fec4-7f89-4805-b427-468307bcaa2d, 7, Finished, Available, Finished, False)

[READ] silver.customers_t
[TEST] customers_t.customer_id_not_null: PASS (failed_record_count=0)
[TEST] customers_t.customer_id_unique: PASS (failed_record_count=0)
[READ] silver.employees_t
[TEST] employees_t.employee_id_not_null: PASS (failed_record_count=0)
[TEST] employees_t.employee_id_unique: PASS (failed_record_count=0)
[READ] silver.orders_t
[TEST] orders_t.order_id_not_null: PASS (failed_record_count=0)
[TEST] orders_t.order_id_unique: PASS (failed_record_count=0)
[READ] silver.order_items_t
[TEST] order_items_t.order_item_id_not_null: PASS (failed_record_count=0)
[TEST] order_items_t.order_item_id_unique: PASS (failed_record_count=0)
[READ] silver.products_t
[TEST] products_t.product_id_not_null: PASS (failed_record_count=0)
[TEST] products_t.product_id_unique: PASS (failed_record_count=0)
[READ] silver.stores_t
[TEST] stores_t.store_id_not_null: PASS (failed_record_count=0)
[TEST] stores_t.store_id_unique: PASS (failed_record_count=0)
[TEST] products_t.product_price_non_negat

In [6]:
print("=" * 80)
print("DATA QUALITY SUMMARY")
print("=" * 80)
display(summary_df)
 
full_results_table = f"{SILVER_SCHEMA}.{DQ_RESULTS_TABLE}"
print(f"[WRITE] {full_results_table}")
 
(
    summary_df.write
      .format("delta")
      .mode("append")          # keep a historical audit log of every run
      .option("mergeSchema", "true")
      .saveAsTable(full_results_table)
)

StatementMeta(, 4da8fec4-7f89-4805-b427-468307bcaa2d, 8, Finished, Available, Finished, False)

DATA QUALITY SUMMARY


SynapseWidget(Synapse.DataFrame, 549564bc-e66d-4822-b882-d5dc2942929e)

[WRITE] silver.dq_test_results


In [7]:
failed_tests = [r for r in dq_results if r["status"] == "FAIL"]
 
if failed_tests:
    failure_lines = "\n".join(
        f"  - {r['table_name']}.{r['test_name']}: {r['failed_record_count']} failed record(s)"
        for r in failed_tests
    )
    error_message = (
        f"Data quality validation FAILED — {len(failed_tests)} test(s) failed:\n"
        f"{failure_lines}\n"
        f"See {full_results_table} for full history."
    )
    print(error_message)
    raise Exception(error_message)
 
print(f"All {len(dq_results)} data quality tests passed.")

StatementMeta(, 4da8fec4-7f89-4805-b427-468307bcaa2d, 9, Finished, Available, Finished, False)

All 20 data quality tests passed.
